# 03 Results Visualization

This notebook produces model comparison figures, class-wise evaluation, confusion matrices, ROC-AUC curves, correlation analysis, feature importance, SHAP interpretation, and spatial prediction visualization.

**Project:** Comparative Study of Machine Learning Models for District-Level Urban Heat Island Classification in Taiwan

**Author:** Zalfa Afifah Zahra

## Cross-validation and model comparison

Visualize cross-validation results and overall model performance for original and SMOTE datasets.

In [ ]:
# =========================================================
# CROSS VALIDATION PERFORMANCE
# ORIGINAL VS SMOTE (SEPARATE PANELS)
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# CV RESULTS
# =========================================================

best_params_df = pd.DataFrame({

    "Dataset":[

        "Original",
        "Original",
        "Original",
        "Original",

        "SMOTE",
        "SMOTE",
        "SMOTE",
        "SMOTE"

    ],

    "Model":[

        "Random Forest",
        "SVM",
        "kNN",
        "XGBoost",

        "Random Forest",
        "SVM",
        "kNN",
        "XGBoost"

    ],

    "Best_F1_CV":[

        0.9016,
        0.5468,
        0.6661,
        0.9095,

        0.8561,
        0.7530,
        0.7744,
        0.9142

    ]

})

# =========================================================
# PREPARE DATA
# =========================================================

cv_plot = best_params_df.pivot(

    index="Model",

    columns="Dataset",

    values="Best_F1_CV"

)

cv_plot = cv_plot.loc[

    [
        "Random Forest",
        "SVM",
        "kNN",
        "XGBoost"
    ]

]

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# COLORS
# =========================================================

blue_palette = [

    "#c6dbef",
    "#9ecae1",
    "#6baed6",
    "#2171b5"

]

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(

    1,
    2,

    figsize=(14,6),

    sharey=True

)

# =========================================================
# DATASETS
# =========================================================

plot_data = [

    ("Original Dataset", "Original"),

    ("SMOTE Dataset", "SMOTE")

]

# =========================================================
# PLOT
# =========================================================

for ax, (title, col) in zip(

    axes,
    plot_data

):

    bars = ax.bar(

        cv_plot.index,

        cv_plot[col],

        color=blue_palette,

        edgecolor="black",

        linewidth=0.8

    )

    # =====================================================
    # VALUE LABELS
    # =====================================================

    for bar in bars:

        height = bar.get_height()

        ax.text(

            bar.get_x() + bar.get_width()/2,

            height + 0.01,

            f"{height:.3f}",

            ha="center",

            va="bottom",

            fontsize=9,

            fontweight="bold"

        )

    # =====================================================
    # AXES
    # =====================================================

    ax.set_title(

        title,

        fontsize=13,

        pad=10

    )

    ax.set_xlabel(

        "Machine Learning Model",

        fontsize=11

    )

    ax.set_xticklabels(

        cv_plot.index,

        rotation=0,

        fontsize=10

    )

    ax.grid(

        axis="y",

        linestyle="--",

        alpha=0.3

    )

    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# =========================================================
# Y LABEL
# =========================================================

axes[0].set_ylabel(

    "Macro F1-Score",

    fontsize=12

)

# =========================================================
# Y LIMIT
# =========================================================

for ax in axes:

    ax.set_ylim(

        0,

        1.0

    )

# =========================================================
# SUPTITLE
# =========================================================

fig.suptitle(

    "Cross-Validation Performance of Tuned Machine Learning Models",

    fontsize=15,

    y=1.02

)

# =========================================================
# LAYOUT
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(

    "Figure_CrossValidation_Original_vs_SMOTE.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# =========================================================
# DISPLAY TABLE
# =========================================================

print("\nCross-Validation F1 Scores")
print("=================================================")

display(

    best_params_df.round(4)

)

print("\nSaved: Figure_CrossValidation_Original_vs_SMOTE.png")

In [ ]:
# =========================================================
# PERFORMANCE COMPARISON
# ORIGINAL VS SMOTE
# =========================================================

import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import MultipleLocator

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# COLORS
# =========================================================

metric_colors = [

    "#c6dbef",   # Accuracy
    "#9ecae1",   # Precision
    "#6baed6",   # Recall
    "#3182bd"    # F1

]

# =========================================================
# MODEL ORDER
# =========================================================

model_order = [

    "XGBoost",
    "Random Forest",
    "kNN",
    "SVM"

]

# =========================================================
# METRICS
# =========================================================

metrics = [

    "Accuracy",
    "Precision",
    "Recall",
    "F1"

]

# =========================================================
# COPY RESULTS
# =========================================================

plot_df = results_df.copy()

# =========================================================
# ORDER MODELS
# =========================================================

plot_df["Model"] = pd.Categorical(

    plot_df["Model"],

    categories=model_order,

    ordered=True

)

# =========================================================
# SPLIT DATA
# =========================================================

plot_original = (

    plot_df[
        plot_df["Dataset"] == "Original"
    ]

    .set_index("Model")

    .reindex(model_order)

)

plot_smote = (

    plot_df[
        plot_df["Dataset"] == "SMOTE"
    ]

    .set_index("Model")

    .reindex(model_order)

)

# =========================================================
# FIGURE
# =========================================================

fig, axes = plt.subplots(

    2,
    1,

    figsize=(16,12),

    sharey=True

)

# =========================================================
# PLOT LOOP
# =========================================================

for ax, data, title in zip(

    axes,

    [
        plot_original,
        plot_smote
    ],

    [
        "Performance Comparison Using Original Dataset",
        "Performance Comparison Using SMOTE Dataset"
    ]

):

    data[metrics].plot(

        kind="bar",

        ax=ax,

        color=metric_colors,

        edgecolor="black",

        linewidth=0.7,

        width=0.85

    )

    # =====================================================
    # REMOVE AUTO LEGEND
    # =====================================================

    if ax.get_legend() is not None:

        ax.get_legend().remove()

    # =====================================================
    # TITLE
    # =====================================================

    ax.set_title(

        title,

        fontsize=15,

        pad=12

    )

    # =====================================================
    # LABELS
    # =====================================================

    ax.set_xlabel(

        "Machine Learning Model",

        fontsize=11

    )

    ax.set_ylabel(

        "Score",

        fontsize=11

    )

    # =====================================================
    # Y LIMIT
    # =====================================================

    ax.set_ylim(

        0.50,
        1.00

    )

    ax.yaxis.set_major_locator(

        MultipleLocator(0.05)

    )

    # =====================================================
    # GRID
    # =====================================================

    ax.grid(

        axis="y",

        linestyle="--",

        alpha=0.4

    )

    ax.set_axisbelow(True)

    # =====================================================
    # TICKS
    # =====================================================

    ax.tick_params(

        axis="x",

        rotation=0,

        labelsize=10

    )

    ax.tick_params(

        axis="y",

        labelsize=10

    )

    # =====================================================
    # REMOVE SPINES
    # =====================================================

    ax.spines["top"].set_visible(False)

    ax.spines["right"].set_visible(False)

    # =====================================================
    # VALUE LABELS
    # =====================================================

    for container in ax.containers:

        labels = [

            f"{value:.3f}"

            for value in container.datavalues

        ]

        ax.bar_label(

            container,

            labels=labels,

            fontsize=8,

            padding=2

        )

# =========================================================
# GLOBAL LEGEND
# =========================================================

handles = [

    plt.Rectangle(

        (0,0),

        1,

        1,

        color=color,

        ec="black"

    )

    for color in metric_colors

]

fig.legend(

    handles,

    metrics,

    title="Evaluation Metrics",

    loc="lower center",

    ncol=4,

    frameon=False,

    fontsize=10,

    bbox_to_anchor=(0.5, -0.01)

)

# =========================================================
# GLOBAL TITLE
# =========================================================

fig.suptitle(

    "Performance Comparison of Machine Learning Models\nOriginal and SMOTE Datasets",

    fontsize=18,

    y=0.98

)

# =========================================================
# LAYOUT
# =========================================================

plt.tight_layout(

    rect=[0,0.05,1,0.95]

)

# =========================================================
# SAVE FIGURE
# =========================================================

plt.savefig(

    "Figure_Model_Performance_Original_vs_SMOTE.png",

    dpi=300,

    bbox_inches="tight"

)

# =========================================================
# SHOW
# =========================================================

plt.show()

# =========================================================
# MESSAGE
# =========================================================

print(
    "\nSaved: Figure_Model_Performance_Original_vs_SMOTE.png"
)

## Class-wise evaluation

Visualize class-specific precision, recall, F1-score, and confusion matrix for the benchmark Random Forest model.

In [ ]:
# =========================================================
# RANDOM FOREST CLASS-WISE PERFORMANCE
# ORIGINAL DATASET ONLY
# BLUE PALETTE BY UHI CLASS
# =========================================================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report

# =========================================================
# UHI CLASS LABELS
# =========================================================

class_mapping = {
    0: "SUCI",
    1: "MUCI",
    2: "TEZ",
    3: "MUHI",
    4: "SUHI"
}

class_order = [
    "SUCI",
    "MUCI",
    "TEZ",
    "MUHI",
    "SUHI"
]

# =========================================================
# BLUE PALETTE (CLASS COLORS)
# =========================================================

class_colors = {
    "SUCI": "#eff3ff",   # Very Light Blue
    "MUCI": "#bdd7e7",   # Light Blue
    "TEZ":  "#6baed6",   # Medium Blue
    "MUHI": "#3182bd",   # Dark Blue
    "SUHI": "#08519c"    # Deep Blue
}

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# FUNCTION
# =========================================================

def plot_rf_classwise_metrics(
    predictions,
    title,
    save_name
):

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================

    rf_predictions = predictions["Random Forest"]

    report = classification_report(
        y_test,
        rf_predictions,
        labels=[0, 1, 2, 3, 4],
        output_dict=True,
        zero_division=0
    )

    # =====================================================
    # BUILD DATAFRAME
    # =====================================================

    records = []

    for class_id, class_name in class_mapping.items():

        records.append({
            "Class": class_name,
            "Precision": report[str(class_id)]["precision"],
            "Recall": report[str(class_id)]["recall"],
            "F1-score": report[str(class_id)]["f1-score"]
        })

    classwise_df = pd.DataFrame(records)

    classwise_df["Class"] = pd.Categorical(
        classwise_df["Class"],
        categories=class_order,
        ordered=True
    )

    classwise_df = classwise_df.sort_values("Class")

    # =====================================================
    # METRICS
    # =====================================================

    metrics = [
        "Precision",
        "Recall",
        "F1-score"
    ]

    # =====================================================
    # FIGURE
    # =====================================================

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 6),
        sharey=True
    )

    # =====================================================
    # PLOTS
    # =====================================================

    for ax, metric in zip(axes, metrics):

        bars = ax.bar(
            classwise_df["Class"],
            classwise_df[metric],
            color=[
                class_colors[c]
                for c in classwise_df["Class"]
            ],
            edgecolor="black",
            linewidth=0.8
        )

        # =================================================
        # VALUE LABELS
        # =================================================

        for bar in bars:

            value = bar.get_height()

            ax.text(
                bar.get_x() + bar.get_width()/2,
                value + 0.01,
                f"{value:.2f}",
                ha="center",
                va="bottom",
                fontsize=9
            )

        # =================================================
        # TITLES
        # =================================================

        ax.set_title(
            metric,
            fontsize=14,
            pad=12
        )

        # =================================================
        # AXES
        # =================================================

        ax.set_xlabel(
            "UHI Class",
            fontsize=11
        )

        ax.tick_params(
            axis="x",
            labelsize=10
        )

        ax.grid(
            axis="y",
            linestyle="--",
            alpha=0.4
        )

        ax.set_axisbelow(True)

        ax.set_ylim(
            0.40,
            1.00
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # =====================================================
    # Y LABEL
    # =====================================================

    axes[0].set_ylabel(
        "Score",
        fontsize=12
    )

    # =====================================================
    # CLASS LEGEND
    # =====================================================

    legend_handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            facecolor=color,
            edgecolor="black"
        )
        for color in class_colors.values()
    ]

    fig.legend(
        legend_handles,
        class_order,
        loc="lower center",
        ncol=5,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, -0.03)
    )

    # =====================================================
    # GLOBAL TITLE
    # =====================================================

    fig.suptitle(
        title,
        fontsize=16,
        y=1.02
    )

    # =====================================================
    # LAYOUT
    # =====================================================

    plt.tight_layout(
        rect=[0, 0.05, 1, 1]
    )

    # =====================================================
    # SAVE
    # =====================================================

    plt.savefig(
        save_name,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return classwise_df

# =========================================================
# GENERATE FIGURE
# =========================================================

print("\nGenerating Random Forest Class-wise Performance Figure...")

rf_classwise_original = plot_rf_classwise_metrics(
    predictions_original,
    "Random Forest Class-wise Performance Using Original Dataset",
    "RF_Classwise_Original.png"
)

# =========================================================
# SAVE CSV
# =========================================================

rf_classwise_original.to_csv(
    "RF_Classwise_Original.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================================================
# DISPLAY RESULTS
# =========================================================

print("\nRANDOM FOREST CLASS-WISE RESULTS")

display(
    rf_classwise_original.round(4)
)

# =========================================================
# FILES
# =========================================================

print("\nSaved Files")
print("✓ RF_Classwise_Original.png")
print("✓ RF_Classwise_Original.csv")

In [ ]:
# =========================================================
# CONFUSION MATRIX
# RANDOM FOREST (ORIGINAL DATASET)
# RAW COUNTS
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# =========================================================
# CLASS LABELS
# =========================================================

class_labels = [
    "SUCI",
    "MUCI",
    "TEZ",
    "MUHI",
    "SUHI"
]

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# BEST MODEL
# =========================================================

rf_model = trained_models["Random Forest_Original"]

# =========================================================
# PREDICTION
# =========================================================

y_pred = rf_model.predict(X_test)

# =========================================================
# CONFUSION MATRIX (RAW COUNTS)
# =========================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1, 2, 3, 4]
)

# =========================================================
# FIGURE
# =========================================================

fig, ax = plt.subplots(
    figsize=(8, 7)
)

# =========================================================
# HEATMAP
# =========================================================

im = ax.imshow(
    cm,
    interpolation="nearest",
    cmap="Blues"
)

# =========================================================
# SHORTER COLORBAR
# =========================================================

cbar = fig.colorbar(
    im,
    ax=ax,
    shrink=0.93,     # shorter colorbar
    pad=0.03
)

cbar.ax.tick_params(
    labelsize=9
)

# =========================================================
# AXES LABELS
# =========================================================

ax.set(
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
    xlabel="Predicted Class",
    ylabel="True Class"
)

ax.tick_params(
    axis="x",
    labelsize=10
)

ax.tick_params(
    axis="y",
    labelsize=10
)

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Confusion Matrix of Random Forest (Original Dataset)",
    fontsize=14,
    pad=15
)

# =========================================================
# ANNOTATIONS
# =========================================================

threshold = cm.max() / 2

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):

        ax.text(
            j,
            i,
            format(cm[i, j], "d"),
            ha="center",
            va="center",
            color="white" if cm[i, j] > threshold else "black",
            fontsize=10
        )

# =========================================================
# CLEAN APPEARANCE
# =========================================================

ax.grid(False)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "RF_Original_ConfusionMatrix_Counts.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# =========================================================
# PRINT MATRIX
# =========================================================

print("\nConfusion Matrix (Raw Counts)")
print(cm)

# =========================================================
# FILES
# =========================================================

print("\nSaved Files")
print("✓ RF_Original_ConfusionMatrix_Counts.png")

In [ ]:
# =========================================================
# MULTICLASS ROC-AUC COMPARISON
# ORIGINAL DATASET VS SMOTE DATASET
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    auc,
    roc_auc_score
)

from sklearn.preprocessing import (
    label_binarize
)

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# CLASSES
# =========================================================

classes = [0, 1, 2, 3, 4]

y_test_bin = label_binarize(

    y_test,

    classes=classes

)

# =========================================================
# MODEL ORDER
# =========================================================

model_order = [

    "Random Forest",
    "SVM",
    "kNN",
    "XGBoost"

]

# =========================================================
# COLORS
# =========================================================

model_colors = {

    "Random Forest": "#c6dbef",

    "SVM": "#9ecae1",

    "kNN": "#6baed6",

    "XGBoost": "#08519c"

}

# =========================================================
# SPLIT MODELS
# =========================================================

best_models_no_smote = {

    "Random Forest":
        trained_models["Random Forest_Original"],

    "SVM":
        trained_models["SVM_Original"],

    "kNN":
        trained_models["kNN_Original"],

    "XGBoost":
        trained_models["XGBoost_Original"]

}

best_models_smote = {

    "Random Forest":
        trained_models["Random Forest_SMOTE"],

    "SVM":
        trained_models["SVM_SMOTE"],

    "kNN":
        trained_models["kNN_SMOTE"],

    "XGBoost":
        trained_models["XGBoost_SMOTE"]

}

# =========================================================
# FUNCTION
# =========================================================

def plot_multiclass_roc(

    fitted_models,
    title,
    save_name

):

    fig, ax = plt.subplots(

        figsize=(9,7)

    )

    auc_table = []

    # =====================================================
    # LOOP MODELS
    # =====================================================

    for model_name in model_order:

        if model_name not in fitted_models:

            continue

        model = fitted_models[model_name]

        # =================================================
        # PROBABILITY PREDICTION
        # =================================================

        try:

            y_score = model.predict_proba(

                X_test

            )

        except Exception:

            print(

                f"Skipping {model_name}: predict_proba unavailable"

            )

            continue

        # =================================================
        # VALIDATE CLASS COUNT
        # =================================================

        if y_score.shape[1] != len(classes):

            print(

                f"Skipping {model_name}: incompatible class count"

            )

            continue

        # =================================================
        # MICRO-AVERAGE ROC
        # =================================================

        fpr, tpr, _ = roc_curve(

            y_test_bin.ravel(),

            y_score.ravel()

        )

        micro_auc = auc(

            fpr,

            tpr

        )

        # =================================================
        # MACRO AUC
        # =================================================

        macro_auc = roc_auc_score(

            y_test,

            y_score,

            multi_class="ovr",

            average="macro"

        )

        auc_table.append({

            "Model":
                model_name,

            "Micro_AUC":
                round(micro_auc, 4),

            "Macro_AUC":
                round(macro_auc, 4)

        })

        # =================================================
        # PLOT
        # =================================================

        ax.plot(

            fpr,

            tpr,

            linewidth=2.5,

            color=model_colors[model_name],

            label=(
                f"{model_name} "
                f"(Macro AUC = {macro_auc:.3f})"
            )

        )

    # =====================================================
    # RANDOM LINE
    # =====================================================

    ax.plot(

        [0,1],

        [0,1],

        linestyle="--",

        color="black",

        linewidth=1.5,

        label="Random Classifier"

    )

    # =====================================================
    # AXES
    # =====================================================

    ax.set_title(

        title,

        fontsize=16,

        pad=15

    )

    ax.set_xlabel(

        "False Positive Rate",

        fontsize=12

    )

    ax.set_ylabel(

        "True Positive Rate",

        fontsize=12

    )

    ax.set_xlim(

        0,

        1

    )

    ax.set_ylim(

        0,

        1.02

    )

    # =====================================================
    # GRID
    # =====================================================

    ax.grid(

        linestyle="--",

        alpha=0.4

    )

    ax.set_axisbelow(True)

    # =====================================================
    # LEGEND
    # =====================================================

    ax.legend(

        loc="lower right",

        fontsize=9

    )

    # =====================================================
    # STYLE
    # =====================================================

    ax.spines["top"].set_visible(False)

    ax.spines["right"].set_visible(False)

    # =====================================================
    # SAVE
    # =====================================================

    plt.tight_layout()

    plt.savefig(

        save_name,

        dpi=300,

        bbox_inches="tight"

    )

    plt.show()

    # =====================================================
    # TABLE
    # =====================================================

    auc_df = pd.DataFrame(

        auc_table

    )

    auc_df = auc_df.sort_values(

        "Macro_AUC",

        ascending=False

    )

    print("\nAUC RESULTS")
    print("=================================================")

    display(

        auc_df

    )

    return auc_df

# =========================================================
# ORIGINAL DATASET
# =========================================================

print(
    "\nGenerating ROC Curve (Original Dataset)..."
)

auc_no_smote = plot_multiclass_roc(

    best_models_no_smote,

    "Multiclass ROC-AUC Comparison (Original Dataset)",

    "Figure_ROC_Original.png"

)

# =========================================================
# SMOTE DATASET
# =========================================================

print(
    "\nGenerating ROC Curve (SMOTE Dataset)..."
)

auc_smote = plot_multiclass_roc(

    best_models_smote,

    "Multiclass ROC-AUC Comparison (SMOTE Dataset)",

    "Figure_ROC_SMOTE.png"

)

# =========================================================
# SAVE TABLES
# =========================================================

auc_no_smote.to_csv(

    "ROC_AUC_Original.csv",

    index=False,

    encoding="utf-8-sig"

)

auc_smote.to_csv(

    "ROC_AUC_SMOTE.csv",

    index=False,

    encoding="utf-8-sig"

)

# =========================================================
# FILES
# =========================================================

print("\nSaved Files")
print("=================================================")

print("✓ Figure_ROC_Original.png")
print("✓ Figure_ROC_SMOTE.png")

print("✓ ROC_AUC_Original.csv")
print("✓ ROC_AUC_SMOTE.csv")

In [ ]:
# =========================================================
# MULTI-CLASS ROC ANALYSIS
# RANDOM FOREST (ORIGINAL DATASET)
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc
)

from sklearn.preprocessing import (
    label_binarize
)

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# CLASS LABELS
# =========================================================

class_labels = {
    0: "SUCI",
    1: "MUCI",
    2: "TEZ",
    3: "MUHI",
    4: "SUHI"
}

classes = np.array([0, 1, 2, 3, 4])

# =========================================================
# BLUE PALETTE
# =========================================================

roc_colors = {
    0: "#eff3ff",
    1: "#bdd7e7",
    2: "#6baed6",
    3: "#3182bd",
    4: "#08519c"
}

# =========================================================
# BINARIZE LABELS
# =========================================================

y_test_bin = label_binarize(
    y_test,
    classes=classes
)

# =========================================================
# BEST MODEL
# =========================================================

rf_model = trained_models["Random Forest_Original"]

# =========================================================
# PREDICT PROBABILITIES
# =========================================================

y_proba = rf_model.predict_proba(X_test)

# =========================================================
# AUC RESULTS
# =========================================================

auc_results = []

for i in classes:

    auc_score = roc_auc_score(
        (y_test == i).astype(int),
        y_proba[:, i]
    )

    auc_results.append({
        "Class": class_labels[i],
        "AUC": round(auc_score, 4)
    })

# =========================================================
# MACRO & MICRO AUC
# =========================================================

macro_auc = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

micro_auc = roc_auc_score(
    y_test_bin,
    y_proba,
    average="micro"
)

# =========================================================
# FIGURE
# =========================================================

fig, ax = plt.subplots(
    figsize=(8, 7)
)

# =========================================================
# ROC CURVES
# =========================================================

for i in classes:

    fpr, tpr, _ = roc_curve(
        y_test_bin[:, i],
        y_proba[:, i]
    )

    class_auc = auc(
        fpr,
        tpr
    )

    ax.plot(
        fpr,
        tpr,
        linewidth=2.5,
        color=roc_colors[i],
        label=(
            f"{class_labels[i]} "
            f"(AUC = {class_auc:.3f})"
        )
    )

# =========================================================
# RANDOM CLASSIFIER
# =========================================================

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="black",
    linewidth=1.2,
    label="Random Classifier"
)

# =========================================================
# FORMATTING
# =========================================================

ax.set_title(
    (
        "One-vs-Rest ROC Curves of Random Forest\n"
        f"Macro-AUC = {macro_auc:.3f} | "
        f"Micro-AUC = {micro_auc:.3f}"
    ),
    fontsize=14,
    pad=15
)

ax.set_xlabel(
    "False Positive Rate",
    fontsize=11
)

ax.set_ylabel(
    "True Positive Rate",
    fontsize=11
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

ax.grid(
    linestyle="--",
    alpha=0.4
)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(
    loc="lower right",
    fontsize=9,
    frameon=True
)

plt.tight_layout()

# =========================================================
# SAVE FIGURE
# =========================================================

plt.savefig(
    "RF_Original_Multiclass_ROC.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# =========================================================
# SAVE TABLE
# =========================================================

auc_table = pd.DataFrame(auc_results)

auc_table.loc[len(auc_table)] = [
    "Macro Average",
    round(macro_auc, 4)
]

auc_table.loc[len(auc_table)] = [
    "Micro Average",
    round(micro_auc, 4)
]

auc_table.to_csv(
    "RF_Original_ROC_AUC_Table.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================================================
# DISPLAY RESULTS
# =========================================================

print("\nROC-AUC RESULTS")
display(auc_table)

# =========================================================
# FILES
# =========================================================

print("\nSaved Files")
print("✓ RF_Original_Multiclass_ROC.png")
print("✓ RF_Original_ROC_AUC_Table.csv")

## Feature analysis and interpretation

Generate correlation matrices, LST relationship analysis, feature importance, and SHAP-based interpretation.

In [ ]:
# =========================================================
# CORRELATION COEFFICIENT ANALYSIS
# UHI DATASET
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "/content/ground_truth_uhi_landsat_5class.csv",
    encoding="utf-8-sig"
)

print("\nDATASET SHAPE")
print("=================================================")
print(df.shape)

# =========================================================
# FEATURES
# =========================================================

features = [

    "AMB_TEMP",
    "RH",
    "WIND_SPEED",

    "Albedo",
    "BSI",
    "EVI",
    "MNDWI",
    "NDBI",
    "NDVI",
    "NDWI",
    "SAVI"
]

# =========================================================
# REMOVE MISSING VALUES
# =========================================================

X = df[features].dropna()

print("\nDATA USED FOR CORRELATION")
print("=================================================")
print(X.shape)

# =========================================================
# PEARSON CORRELATION
# =========================================================

corr_matrix = X.corr(
    method="pearson"
)

print("\nPEARSON CORRELATION MATRIX")
print("=================================================")

display(
    corr_matrix.round(3)
)

# =========================================================
# SAVE TABLE
# =========================================================

corr_matrix.round(4).to_csv(

    "Correlation_Matrix_UHI.csv",

    index=True,

    encoding="utf-8-sig"
)

# =========================================================
# HEATMAP
# =========================================================

fig, ax = plt.subplots(

    figsize=(12,10)
)

im = ax.imshow(

    corr_matrix,

    cmap="Blues",

    vmin=-1,

    vmax=1,

    aspect="auto"
)

# =========================================================
# TICKS
# =========================================================

ax.set_xticks(
    range(len(features))
)

ax.set_yticks(
    range(len(features))
)

ax.set_xticklabels(

    features,

    rotation=90,

    fontsize=10
)

ax.set_yticklabels(

    features,

    fontsize=10
)

# =========================================================
# CORRELATION VALUES
# =========================================================

for i in range(len(features)):

    for j in range(len(features)):

        ax.text(

            j,

            i,

            f"{corr_matrix.iloc[i,j]:.2f}",

            ha="center",

            va="center",

            fontsize=7
        )

# =========================================================
# COLORBAR
# =========================================================

cbar = plt.colorbar(im)

cbar.set_label(

    "Pearson Correlation Coefficient (r)",

    fontsize=11
)

# =========================================================
# TITLE
# =========================================================

plt.title(

    "Correlation Matrix of Meteorological and Remote Sensing Variables",

    fontsize=18,

    pad=15
)

# =========================================================
# SAVE FIGURE
# =========================================================

plt.tight_layout()

plt.savefig(

    "Figure_Correlation_Heatmap_UHI.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

# =========================================================
# STRONG CORRELATIONS
# =========================================================

print("\nSTRONG CORRELATIONS (|r| ≥ 0.70)")
print("=================================================")

strong_corr = []

for i in range(len(features)):

    for j in range(i + 1, len(features)):

        corr_value = corr_matrix.iloc[i, j]

        if abs(corr_value) >= 0.70:

            strong_corr.append([

                features[i],

                features[j],

                round(corr_value, 4)
            ])

strong_corr_df = pd.DataFrame(

    strong_corr,

    columns=[

        "Feature_1",

        "Feature_2",

        "Correlation"
    ]
)

if len(strong_corr_df) > 0:

    display(
        strong_corr_df.sort_values(
            by="Correlation",
            ascending=False
        )
    )

    strong_corr_df.to_csv(

        "Strong_Correlations_UHI.csv",

        index=False,

        encoding="utf-8-sig"
    )

else:

    print(
        "No strong correlations found."
    )

# =========================================================
# FILES
# =========================================================

print("\nFILES SAVED")
print("=================================================")

print("✓ Correlation_Matrix_UHI.csv")
print("✓ Figure_Correlation_Heatmap_UHI.png")
print("✓ Strong_Correlations_UHI.csv")

In [ ]:
# =========================================================
# CORRELATION WITH LAND SURFACE TEMPERATURE (LST)
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(
    "/content/ground_truth_uhi_landsat_5class.csv",
    encoding="utf-8-sig"
)

# =========================================================
# FEATURE LIST
# =========================================================

features = [

    "AMB_TEMP",
    "RH",
    "WIND_SPEED",

    "Albedo",
    "BSI",
    "EVI",
    "MNDWI",
    "NDBI",
    "NDVI",
    "NDWI",
    "SAVI"
]

# =========================================================
# TARGET
# =========================================================

target = "LST_C"

# =========================================================
# REMOVE MISSING VALUES
# =========================================================

df_corr = df[
    features + [target]
].dropna()

# =========================================================
# CORRELATION ANALYSIS
# =========================================================

target_corr = (

    df_corr[
        features + [target]
    ]

    .corr(method="pearson")[target]

    .drop(target)

    .sort_values(
        key=np.abs,
        ascending=False
    )
)

# =========================================================
# TABLE
# =========================================================

corr_table = pd.DataFrame({

    "Feature":
        target_corr.index,

    "Correlation":
        target_corr.values
})

corr_table = corr_table.round(4)

# =========================================================
# DISPLAY
# =========================================================

print("\nCORRELATION WITH LST")
print("=================================================")

display(corr_table)

# =========================================================
# SAVE TABLE
# =========================================================

corr_table.to_csv(

    "LST_Feature_Correlation.csv",

    index=False,

    encoding="utf-8-sig"
)

# =========================================================
# BAR CHART
# =========================================================

fig, ax = plt.subplots(

    figsize=(10,7)
)

colors = [

    "#08519c"
    if x >= 0
    else "#9ecae1"

    for x in corr_table["Correlation"]
]

bars = ax.barh(

    corr_table["Feature"],

    corr_table["Correlation"],

    color=colors,

    edgecolor="black"
)

# =========================================================
# VALUE LABELS
# =========================================================

for bar in bars:

    value = bar.get_width()

    ax.text(

        value/2,

        bar.get_y() + bar.get_height()/2,

        f"{value:.3f}",

        ha="center",

        va="center",

        fontsize=9,

        color="white",

        fontweight="bold"
    )

# =========================================================
# AXES
# =========================================================

ax.set_xlabel(

    "Pearson Correlation Coefficient (r)",

    fontsize=12
)

ax.set_ylabel(

    "Variable",

    fontsize=12
)

ax.set_title(

    "Correlation Between Input Variables and Land Surface Temperature",

    fontsize=16,

    pad=12
)

ax.axvline(

    x=0,

    linestyle="--",

    linewidth=1,

    color="black"
)

ax.grid(

    axis="x",

    linestyle="--",

    alpha=0.3
)

ax.set_axisbelow(True)

ax.invert_yaxis()

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# =========================================================
# SAVE FIGURE
# =========================================================

plt.tight_layout()

plt.savefig(

    "Figure_LST_Correlation.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

# =========================================================
# TOP POSITIVE CORRELATIONS
# =========================================================

print("\nTOP POSITIVE CORRELATIONS")
print("=================================================")

display(

    target_corr
    .sort_values(ascending=False)
    .head(5)
    .round(4)
)

# =========================================================
# TOP NEGATIVE CORRELATIONS
# =========================================================

print("\nTOP NEGATIVE CORRELATIONS")
print("=================================================")

display(

    target_corr
    .sort_values()
    .head(5)
    .round(4)
)

# =========================================================
# SAVE STRONG CORRELATIONS
# =========================================================

strong_corr = corr_table[
    abs(corr_table["Correlation"]) >= 0.50
]

strong_corr.to_csv(

    "Strong_LST_Correlations.csv",

    index=False,

    encoding="utf-8-sig"
)

# =========================================================
# FILES
# =========================================================

print("\nFILES SAVED")
print("=================================================")

print("✓ LST_Feature_Correlation.csv")
print("✓ Figure_LST_Correlation.png")
print("✓ Strong_LST_Correlations.csv")

In [ ]:
# =========================================================
# FEATURE IMPORTANCE
# RANDOM FOREST (ORIGINAL DATASET)
# UHI CLASSIFICATION
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# STYLE
# =========================================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# =========================================================
# FEATURES
# =========================================================

feature_names = [

    # Meteorological Variables
    "AMB_TEMP",
    "RH",
    "WIND_SPEED",

    # Remote Sensing Variables
    "Albedo",
    "BSI",
    "EVI",
    "MNDWI",
    "NDBI",
    "NDVI",
    "NDWI",
    "SAVI"

]

# =========================================================
# BEST MODEL
# =========================================================

rf_model = trained_models["Random Forest_Original"]

# =========================================================
# PIPELINE HANDLING
# =========================================================

if hasattr(rf_model, "named_steps"):

    if "model" in rf_model.named_steps:

        estimator = rf_model.named_steps["model"]

    else:

        estimator = list(
            rf_model.named_steps.values()
        )[-1]

else:

    estimator = rf_model

# =========================================================
# FEATURE IMPORTANCE
# =========================================================

importance_df = pd.DataFrame({

    "Feature": feature_names,

    "Importance":
        estimator.feature_importances_

})

importance_df = (

    importance_df

    .sort_values(
        by="Importance",
        ascending=False
    )

    .reset_index(drop=True)

)

# =========================================================
# SAVE TABLE
# =========================================================

importance_df.to_csv(

    "RF_Original_FeatureImportance.csv",

    index=False,

    encoding="utf-8-sig"

)

# =========================================================
# FIGURE
# =========================================================

fig, ax = plt.subplots(

    figsize=(9, 6)

)

# =========================================================
# BLUE PALETTE
# =========================================================

colors = plt.cm.Blues(

    np.linspace(
        0.35,
        0.90,
        len(importance_df)
    )

)

# =========================================================
# HORIZONTAL BAR CHART
# =========================================================

bars = ax.barh(

    importance_df["Feature"][::-1],

    importance_df["Importance"][::-1],

    color=colors[::-1],

    edgecolor="black",

    linewidth=0.7

)

# =========================================================
# VALUE LABELS
# =========================================================

offset = (

    importance_df["Importance"].max()

    * 0.03

)

for i, value in enumerate(

    importance_df["Importance"][::-1]

):

    ax.text(

        value + offset,

        i,

        f"{value:.3f}",

        fontsize=9,

        va="center"

    )

# =========================================================
# STYLE
# =========================================================

ax.set_title(

    "Feature Importance of Random Forest (Original Dataset)",

    fontsize=14,


    pad=12

)

ax.set_xlabel(

    "Importance Score",

    fontsize=11

)

ax.set_ylabel(

    "Predictor Variables",

    fontsize=11

)

ax.grid(

    axis="x",

    linestyle="--",

    alpha=0.4

)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)

ax.spines["right"].set_visible(False)

# =========================================================
# LAYOUT
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE FIGURE
# =========================================================

plt.savefig(

    "RF_Original_FeatureImportance.png",

    dpi=300,

    bbox_inches="tight"

)

plt.show()

# =========================================================
# DISPLAY TABLE
# =========================================================

print("\nRANDOM FOREST FEATURE IMPORTANCE")
display(

    importance_df.round(4)

)

# =========================================================
# FILES
# =========================================================

print("\nSaved Files")
print("✓ RF_Original_FeatureImportance.png")
print("✓ RF_Original_FeatureImportance.csv")

In [ ]:
# =========================================================
# SHAP SUMMARY PLOT
# BEST XGBOOST MODEL
# =========================================================

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# BEST MODEL
# =========================================================

best_xgb = fitted_models_smote["XGBoost"]

# =========================================================
# FEATURES
# =========================================================

feature_names = [

    # Meteorological Variables
    "AMB_TEMP",
    "RH",
    "WIND_SPEED",

    # Remote Sensing Variables
    "Albedo",
    "BSI",
    "EVI",
    "MNDWI",
    "NDBI",
    "NDVI",
    "NDWI",
    "SAVI"

]

# =========================================================
# SAMPLE DATA
# =========================================================

X_shap = X_test.copy()

# =========================================================
# SHAP EXPLAINER
# =========================================================

explainer = shap.TreeExplainer(
    best_xgb
)

shap_values = explainer.shap_values(
    X_shap
)

# =========================================================
# MULTICLASS HANDLING
# =========================================================

if isinstance(shap_values, list):

    shap_summary = np.mean(
        np.abs(
            np.stack(shap_values)
        ),
        axis=0
    )

else:

    shap_summary = np.mean(
        np.abs(shap_values),
        axis=2
    )

# =========================================================
# SUMMARY PLOT
# =========================================================

plt.figure(figsize=(10,7))

shap.summary_plot(

    shap_summary,

    X_shap,

    feature_names=feature_names,

    show=False
)

plt.title(
    "SHAP Summary Plot for UHI Classification",
    fontsize=15
)

plt.tight_layout()

plt.savefig(

    "Figure_SHAP_Summary.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

print(
    "Saved: Figure_SHAP_Summary.png"
)

In [ ]:
# =========================================================
# SHAP IMPORTANCE TABLE
# =========================================================

shap_importance = pd.DataFrame({

    "Feature": feature_names,

    "Mean_Abs_SHAP":

        np.mean(

            np.abs(shap_summary),

            axis=0
        )

})

shap_importance = (

    shap_importance

    .sort_values(

        by="Mean_Abs_SHAP",

        ascending=False
    )

)

display(
    shap_importance.round(4)
)

shap_importance.to_csv(

    "SHAP_Importance_Table.csv",

    index=False
)

print(
    "Saved: SHAP_Importance_Table.csv"
)

In [ ]:
# =========================================================
# SHAP DEPENDENCE PLOT
# NDVI
# =========================================================

shap.dependence_plot(

    "NDVI",

    shap_summary,

    X_shap,

    interaction_index=None,

    show=False
)

plt.title(
    "SHAP Dependence Plot - NDVI"
)

plt.tight_layout()

plt.savefig(

    "Figure_SHAP_NDVI.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# SHAP DEPENDENCE PLOT
# NDBI
# =========================================================

shap.dependence_plot(

    "NDBI",

    shap_summary,

    X_shap,

    interaction_index=None,

    show=False
)

plt.title(
    "SHAP Dependence Plot - NDBI"
)

plt.tight_layout()

plt.savefig(

    "Figure_SHAP_NDBI.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# SHAP DEPENDENCE PLOT
# AMB_TEMP
# =========================================================

shap.dependence_plot(

    "AMB_TEMP",

    shap_summary,

    X_shap,

    interaction_index=None,

    show=False
)

plt.title(
    "SHAP Dependence Plot - Ambient Temperature"
)

plt.tight_layout()

plt.savefig(

    "Figure_SHAP_AMB_TEMP.png",

    dpi=300,

    bbox_inches="tight"
)

plt.show()

## Spatial prediction

Apply the trained model to spatial data and visualize UHI prediction maps.

In [ ]:
# =========================================================
# SPATIAL PREDICTION - RANDOM FOREST ORIGINAL
# TAIPEI & HSINCHU (2020, 2023, 2026)
# OUTPUT: GEOtiff FOR QGIS
# =========================================================

import os
import joblib
import numpy as np
import pandas as pd
import rasterio

# =========================================================
# LOAD MODEL
# =========================================================

model = joblib.load("/content/Random_Forest_Original_best.pkl")
print("✓ Model loaded: Random Forest Original")

# =========================================================
# TRAINING FEATURES (FOLLOW YOUR MODEL)
# =========================================================
features = [

    # Meteorological Variables
    "AMB_TEMP",
    "RH",
    "WIND_SPEED",

    # Remote Sensing Variables
    "Albedo",
    "BSI",
    "EVI",
    "MNDWI",
    "NDBI",
    "NDVI",
    "NDWI",
    "SAVI"

]

n_features = len(features)

# =========================================================
# INPUT TIFF FILES (MULTI CITY + YEAR)
# =========================================================

target_files = [
    "/content/Taipei_2021_RF_Features.tif",
    "/content/Taipei_2022_RF_Features.tif",
    "/content/Taipei_2024_RF_Features.tif",
    "/content/Taipei_2025_RF_Features.tif",

]

# =========================================================
# HELPER: PARSE NAME
# =========================================================

def parse_filename(path):
    name = os.path.basename(path).replace(".tif", "")
    parts = name.split("_")
    city = parts[0]
    year = parts[1]
    return city, year

# =========================================================
# PREDICTION FUNCTION
# =========================================================

def predict_tiff(tiff_path):

    with rasterio.open(tiff_path) as src:
        image = src.read()
        profile = src.profile.copy()

    # reshape: (bands, H, W) -> (H, W, bands)
    stack = np.moveaxis(image, 0, -1)

    h, w, b = stack.shape

    if b != n_features:
        raise ValueError(
            f"Band mismatch! TIFF has {b} bands but model expects {n_features}"
        )

    X = stack.reshape(-1, b)

    valid_mask = np.isfinite(X).all(axis=1)

    X_valid = pd.DataFrame(X[valid_mask], columns=features)

    pred = model.predict(X_valid)

    pred_map = np.full(h * w, np.nan, dtype=np.float32)
    pred_map[valid_mask] = pred
    pred_map = pred_map.reshape(h, w)

    return pred_map, profile

# =========================================================
# RUN PREDICTIONS
# =========================================================

for tif in target_files:

    city, year = parse_filename(tif)

    print(f"\nProcessing: {city} - {year}")

    pred_map, profile = predict_tiff(tif)

    # =====================================================
    # SAVE GEOTIFF (QGIS READY)
    # =====================================================

    output_tif = f"/content/{city}_{year}_RF_Prediction.tif"

    save_profile = profile.copy()
    save_profile.update(
        dtype=rasterio.uint8,
        count=1,
        nodata=255,
        compress="lzw"
    )

    pred_save = np.where(
        np.isnan(pred_map),
        255,
        pred_map
    ).astype(np.uint8)

    with rasterio.open(output_tif, "w", **save_profile) as dst:
        dst.write(pred_save, 1)

    print(f"✓ Saved: {output_tif}")

print("\n🎯 All predictions completed (QGIS-ready GeoTIFFs)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Patch

# ==========================================
# JOURNAL STYLE
# ==========================================

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Liberation Serif"]

# ==========================================
# FUNCTION
# ==========================================

def plot_spatial_prediction(
    reservoir_name,
    low_file,
    high_file,
    save_name
):

    low_img = mpimg.imread(low_file)
    high_img = mpimg.imread(high_file)

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(10, 5)
    )

    # ======================================
    # LOW BLOOM
    # ======================================

    axes[0].imshow(low_img)

    axes[0].set_title(
        "(a) Low Bloom Period",
        fontsize=12,
    )

    axes[0].set_xticks([])
    axes[0].set_yticks([])

    # ======================================
    # HIGH BLOOM
    # ======================================

    axes[1].imshow(high_img)

    axes[1].set_title(
        "(b) High Bloom Period",
        fontsize=12,
    )

    axes[1].set_xticks([])
    axes[1].set_yticks([])

    # ======================================
    # BORDERS
    # ======================================

    for ax in axes:

        for spine in ax.spines.values():

            spine.set_visible(True)
            spine.set_linewidth(0.8)
            spine.set_color("black")

    fig.suptitle(
        f"{reservoir_name} Reservoir",
        fontsize=15,
        y=0.95
    )

    # ======================================
    # LEGEND
    # ======================================

    legend_elements = [
        Patch(facecolor="green", edgecolor="black",
              label="Class 0: Low Bloom Potential (< 10 μg/L)"),

        Patch(facecolor="yellow", edgecolor="black",
              label="Class 1: Moderate Productivity (10–75 μg/L)"),

        Patch(facecolor="red", edgecolor="black",
              label="Class 2: Elevated Chlorophyll (> 75 μg/L)")
    ]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=3,
        fontsize=10,
        frameon=True,
        bbox_to_anchor=(0.5, -0.02)
    )

    plt.tight_layout(rect=[0, 0.08, 1, 1])

    plt.savefig(
        save_name,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    print(f"Saved: {save_name}")

# ==========================================
# LIYUTAN
# ==========================================

plot_spatial_prediction(
    reservoir_name="Liyutan",
    low_file="/content/Liyutan_Low.jpeg",
    high_file="/content/Liyutan_High.jpeg",
    save_name="Figure_4_19_Liyutan_Spatial_Prediction.jpeg"
)

# ==========================================
# MINGDE
# ==========================================

plot_spatial_prediction(
    reservoir_name="Mingde",
    low_file="/content/Mingde_Low.jpeg",
    high_file="/content/Mingde_High.jpeg",
    save_name="Figure_4_20_Mingde_Spatial_Prediction.jpeg"
)